# Galaxy Sounds Recorders — EDA\n\nProducts: `GALAXY_SOUNDS_BLACK_HOLES`, `GALAXY_SOUNDS_DARK_MATTER`, `GALAXY_SOUNDS_PLANETARY_RINGS`, `GALAXY_SOUNDS_SOLAR_FLAMES`, `GALAXY_SOUNDS_SOLAR_WINDS`  \nDays: 2, 3, 4 · Position limit: ±10 per instrument  \nGoal: identify exploitable market inefficiencies to maximise P&L"

In [ ]:
import warnings, itertools
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import signal, stats
from scipy.fft import fft, fftfreq
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf, ccf, grangercausalitytests, coint
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.stats.diagnostic import acorr_ljungbox
import statsmodels.api as sm
import ruptures as rpt
from arch import arch_model

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})
COLORS = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2"]
INSTRUMENTS = [
    "GALAXY_SOUNDS_BLACK_HOLES",
    "GALAXY_SOUNDS_DARK_MATTER",
    "GALAXY_SOUNDS_PLANETARY_RINGS",
    "GALAXY_SOUNDS_SOLAR_FLAMES",
    "GALAXY_SOUNDS_SOLAR_WINDS",
]
SHORT = {i: i.replace("GALAXY_SOUNDS_","") for i in INSTRUMENTS}
DATA_DIR = "../Data_ROUND_5"
DAYS = [2, 3, 4]
print("Imports OK")

## 1. Data Loading & Quality Check"

In [ ]:
# ── Load prices ──────────────────────────────────────────────────────────────
price_frames = []
for day in DAYS:
    df = pd.read_csv(f"{DATA_DIR}/prices_round_5_day_{day}.csv", sep=";")
    df["day"] = day
    df["global_ts"] = (day - 2) * 1_000_000 + df["timestamp"]
    price_frames.append(df)

prices_raw = pd.concat(price_frames, ignore_index=True)
prices_raw.columns = [c.strip() for c in prices_raw.columns]

gs = prices_raw[prices_raw["product"].isin(INSTRUMENTS)].copy()
gs = gs.sort_values(["product","global_ts"]).reset_index(drop=True)

# ── Load trades ───────────────────────────────────────────────────────────────
trade_frames = []
for day in DAYS:
    df = pd.read_csv(f"{DATA_DIR}/trades_round_5_day_{day}.csv", sep=";")
    df["day"] = day
    df["global_ts"] = (day - 2) * 1_000_000 + df["timestamp"]
    trade_frames.append(df)

trades_raw = pd.concat(trade_frames, ignore_index=True)
trades_raw.columns = [c.strip() for c in trades_raw.columns]
gt = trades_raw[trades_raw["symbol"].isin(INSTRUMENTS)].copy()
gt = gt.sort_values(["symbol","global_ts"]).reset_index(drop=True)

print(f"Price rows (Galaxy Sounds): {len(gs):,}   |   Trade rows: {len(gt):,}")
print("\nPrice columns:", list(gs.columns))
print("\nTrade columns:", list(gt.columns))
print("\nMissing values in prices:\n", gs[["mid_price","bid_price_1","ask_price_1"]].isnull().sum())
print("\nTicks per product per day:")
gs.groupby(["product","day"]).size().unstack()

In [ ]:
# ── Build per-instrument mid-price series (indexed by global_ts) ──────────────
mid = gs.pivot_table(index="global_ts", columns="product", values="mid_price")
mid = mid[INSTRUMENTS]
mid.columns = [SHORT[c] for c in mid.columns]

spread_bbo = gs.copy()
spread_bbo["bbo_spread"] = spread_bbo["ask_price_1"] - spread_bbo["bid_price_1"]

# ── Log-returns ───────────────────────────────────────────────────────────────
log_ret = np.log(mid).diff().dropna()

print("Mid-price descriptive stats:")
mid.describe().round(2)

## 2. Mid-Price Overview — All Instruments on One Chart\n\nThis is the entry point: plot all five mid-prices on a shared axis to get an immediate visual impression of level, drift, co-movement, and any visible structural breaks."

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
for col, c in zip(mid.columns, COLORS):
    ax.plot(mid.index, mid[col], lw=0.7, alpha=0.85, label=col, color=c)

# shade day boundaries
for d in [1, 2]:
    ax.axvline(d * 1_000_000, color="k", lw=0.8, ls="--", alpha=0.4)
    ax.text(d * 1_000_000 + 5000, ax.get_ylim()[0], f" Day {d+2}", fontsize=8, color="k", alpha=0.6)

ax.set_title("Galaxy Sounds — Mid-Price (all 3 days)", fontsize=13)
ax.set_xlabel("Global timestamp (ticks)")
ax.set_ylabel("Mid-price (XIRECS)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

print("""
Interpretation:
  • All five instruments share a similar price level (~10 000 XIRECS) and clearly
    co-move — consistent with them being variants of the same underlying 'product family'.
  • Visual drift and any divergences between instruments are signals for pair-trading.
  • Day boundaries (dashed lines) let us check for overnight gaps or regime changes.
""")

## 3. Trend & Drift — Rolling Means + Hurst Exponent + ADF Unit-Root Test"

In [ ]:
WINDOWS = [20, 50, 100]
fig, axes = plt.subplots(5, 1, figsize=(16, 18), sharex=True)

for ax, col, c in zip(axes, mid.columns, COLORS):
    s = mid[col].dropna()
    ax.plot(s.index, s.values, lw=0.6, color=c, alpha=0.6, label="raw")
    win_colors = ["navy", "darkorange", "green"]
    for w, wc in zip(WINDOWS, win_colors):
        rm = s.rolling(w).mean()
        ax.plot(rm.index, rm.values, lw=1.2, color=wc, label=f"MA-{w}")
    ax.set_title(col, fontsize=10)
    ax.legend(loc="upper right", fontsize=7, ncol=4)

axes[-1].set_xlabel("Global timestamp")
fig.suptitle("Rolling Means (20 / 50 / 100 ticks) vs Raw Mid-Price", fontsize=13, y=1.001)
plt.tight_layout()
plt.show()

print("""
Interpretation:
  • Where the price consistently tracks above its MA → persistent uptrend (Hurst > 0.5).
  • Where price oscillates symmetrically around the MA → mean-reversion (Hurst < 0.5).
  • MA-crossovers (short over long) can seed a momentum signal.
""")

In [ ]:
def hurst_rs(ts, min_n=8):
    """R/S rescaled-range Hurst exponent estimator."""
    ts = np.array(ts, dtype=float)
    ts = ts[~np.isnan(ts)]
    N = len(ts)
    lags, rs_vals = [], []
    for n in [int(N / k) for k in [2, 4, 8, 16, 32, 64] if N // k >= min_n]:
        rs_chunk = []
        for start in range(0, N - n + 1, n):
            sub = ts[start:start+n]
            sub = sub - sub.mean()
            cumdev = np.cumsum(sub)
            R = cumdev.max() - cumdev.min()
            S = sub.std(ddof=1)
            if S > 0:
                rs_chunk.append(R / S)
        if rs_chunk:
            lags.append(np.log(n))
            rs_vals.append(np.log(np.mean(rs_chunk)))
    if len(lags) < 3:
        return np.nan
    slope, *_ = np.polyfit(lags, rs_vals, 1)
    return slope

results = []
for col in mid.columns:
    s = mid[col].dropna()
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(s, autolag="AIC")
    H = hurst_rs(s.values)
    results.append({
        "Instrument": col,
        "ADF stat": round(adf_stat, 3),
        "ADF p-value": round(adf_p, 4),
        "ADF stationarity": "stationary ✓" if adf_p < 0.05 else "unit root ✗",
        "Hurst (R/S)": round(H, 3),
        "Process type": "mean-reverting" if H < 0.45 else ("trending" if H > 0.55 else "random walk"),
    })

res_df = pd.DataFrame(results).set_index("Instrument")
print(res_df.to_string())
print("""
Hurst guide: H < 0.5 → mean-reverting  |  H ≈ 0.5 → random walk  |  H > 0.5 → trending
ADF p < 0.05 → reject unit root → price is stationary (mean-reverts around a level).
""")

## 4. Autocorrelation — ACF / PACF + ADF + KPSS\n\nACF measures how much today's price is correlated with lags of itself. PACF isolates the direct effect at each lag. Together they identify the lag structure needed for a predictive model and confirm whether price or returns are exploitably autocorrelated."

In [ ]:
NLAGS = 40
fig, axes = plt.subplots(5, 2, figsize=(16, 18))

for i, col in enumerate(mid.columns):
    ret = log_ret[col].dropna()
    plot_acf(ret, lags=NLAGS, ax=axes[i][0], title=f"ACF — {col} log-returns", zero=False)
    plot_pacf(ret, lags=NLAGS, ax=axes[i][1], title=f"PACF — {col} log-returns", zero=False, method="ywm")

plt.tight_layout()
plt.show()

# ADF + KPSS on log-returns
stat_table = []
for col in mid.columns:
    ret = log_ret[col].dropna()
    adf_s, adf_p, adf_lag, *_ = adfuller(ret, autolag="AIC")
    try:
        kpss_s, kpss_p, kpss_lag, _ = kpss(ret, regression="c", nlags="auto")
        kpss_str = f"{kpss_p:.4f} ({'stationary ✓' if kpss_p > 0.05 else 'nonstat ✗'})"
    except Exception:
        kpss_str = "n/a"
    acf_vals = acf(ret, nlags=NLAGS, fft=True)
    ci = 1.96 / np.sqrt(len(ret))
    first_insig = next((l for l, v in enumerate(acf_vals[1:], 1) if abs(v) < ci), NLAGS)
    stat_table.append({
        "Instrument": col,
        "ADF p (returns)": f"{adf_p:.4f}",
        "ADF lags": adf_lag,
        "KPSS p (returns)": kpss_str,
        "1st insig ACF lag": first_insig,
    })

pd.DataFrame(stat_table).set_index("Instrument")

## 5. Seasonality & Cycles — FFT Periodogram + STL Decomposition\n\nFFT identifies dominant periodic components in the price series. STL separates trend, seasonal, and residual components to reveal whether any cyclical structure is exploitable."

In [ ]:
# ── FFT Periodogram ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(5, 1, figsize=(16, 14), sharex=False)

for ax, col, c in zip(axes, mid.columns, COLORS):
    s = mid[col].ffill().values.astype(float)
    s -= s.mean()
    N = len(s)
    freqs = fftfreq(N, d=1)[1:N//2]
    power = np.abs(fft(s)[1:N//2])**2
    periods = 1.0 / freqs
    # plot vs period (ticks)
    ax.semilogy(periods, power, lw=0.7, color=c)
    # annotate top-3 peaks
    top_idx = np.argsort(power)[-3:][::-1]
    for idx in top_idx:
        ax.axvline(periods[idx], ls="--", lw=0.8, alpha=0.6)
        ax.text(periods[idx], power[idx], f"  T={periods[idx]:.0f}", fontsize=7)
    ax.set_title(f"FFT — {col}", fontsize=9)
    ax.set_xlabel("Period (ticks)")
    ax.set_ylabel("Power")
    ax.set_xlim(2, 1000)

plt.tight_layout()
plt.show()

print("""
Interpretation:
  • Sharp spectral peaks → exploitable periodicity (e.g. a market-maker resets every T ticks).
  • Broad, flat spectrum → noise-dominated; no cyclical edge.
  • If dominant period T matches your signal window, use T as your look-back.
""")

In [ ]:
# ── STL Decomposition ─────────────────────────────────────────────────────────
# Use period=100 ticks as a reasonable intra-day cycle guess;
# adjust if FFT revealed a different dominant period.
STL_PERIOD = 100
fig, axes = plt.subplots(5, 3, figsize=(18, 16))

for i, (col, c) in enumerate(zip(mid.columns, COLORS)):
    s = mid[col].ffill()
    stl = STL(s, period=STL_PERIOD, robust=True).fit()
    axes[i][0].plot(stl.trend.index, stl.trend.values, lw=0.7, color=c)
    axes[i][0].set_title(f"{col}\nTrend", fontsize=8)
    axes[i][1].plot(stl.seasonal.index, stl.seasonal.values, lw=0.5, color="steelblue")
    axes[i][1].set_title("Seasonal", fontsize=8)
    axes[i][2].plot(stl.resid.index, stl.resid.values, lw=0.5, color="gray")
    axes[i][2].axhline(0, color="k", lw=0.5)
    axes[i][2].set_title("Residual", fontsize=8)

plt.suptitle(f"STL Decomposition (period={STL_PERIOD} ticks)", fontsize=12, y=1.001)
plt.tight_layout()
plt.show()

print("""
Interpretation:
  • Trend component: the slow-moving fair value. Deviations from it are potential entries.
  • Seasonal component: intra-period cycle. If large and consistent, it can be traded directly.
  • Residual: white noise ideally. Autocorrelated residuals → STL period is mis-specified.
""")

## 6. Cross-Instrument Correlation — Pearson, Spearman & Rolling 50-tick Correlation"

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pearson = log_ret.corr(method="pearson")
spearman = log_ret.corr(method="spearman")

for ax, mat, title in zip(axes, [pearson, spearman], ["Pearson", "Spearman"]):
    mask = np.triu(np.ones_like(mat, dtype=bool), k=1)
    sns.heatmap(mat, ax=ax, annot=True, fmt=".3f", cmap="RdYlGn",
                vmin=-1, vmax=1, square=True,
                xticklabels=[SHORT[c+"_".join([""])] if "GALAXY" not in c else c.replace("GALAXY_SOUNDS_","") for c in INSTRUMENTS],
                yticklabels=[c.replace("GALAXY_SOUNDS_","") for c in INSTRUMENTS])
    ax.set_title(f"{title} correlation of log-returns", fontsize=11)

plt.tight_layout()
plt.show()

print("Pairs with |Pearson corr| > 0.7 on returns (potential clusters / hedges):")
pairs_high = [(c1, c2, round(pearson.loc[c1, c2], 3))
              for c1, c2 in itertools.combinations(mid.columns, 2)
              if abs(pearson.loc[c1, c2]) > 0.7]
if pairs_high:
    for c1, c2, v in pairs_high:
        print(f"  {c1}  ↔  {c2}  :  {v}")
else:
    print("  None above 0.7")

In [ ]:
# Rolling 50-tick correlation for every pair
pairs = list(itertools.combinations(mid.columns, 2))
n_pairs = len(pairs)
fig, axes = plt.subplots(n_pairs, 1, figsize=(16, 3.5 * n_pairs), sharex=True)
if n_pairs == 1:
    axes = [axes]

for ax, (c1, c2) in zip(axes, pairs):
    roll_corr = log_ret[c1].rolling(50).corr(log_ret[c2])
    ax.plot(roll_corr.index, roll_corr.values, lw=0.7)
    ax.axhline(0.7, color="red", ls="--", lw=0.8, alpha=0.7, label="|corr|=0.7")
    ax.axhline(-0.7, color="red", ls="--", lw=0.8, alpha=0.7)
    ax.axhline(0, color="k", lw=0.4)
    ax.set_title(f"Rolling 50-tick corr: {c1.replace('GALAXY_SOUNDS_','')} ↔ {c2.replace('GALAXY_SOUNDS_','')}", fontsize=9)
    ax.set_ylim(-1.05, 1.05)
    ax.legend(loc="upper right", fontsize=7)

axes[-1].set_xlabel("Global timestamp")
plt.suptitle("Rolling 50-tick Correlation Between Instruments", fontsize=12, y=1.001)
plt.tight_layout()
plt.show()

print("""
Interpretation:
  • Persistently high rolling correlation (>0.7) → instruments are a near-perfect cluster;
    one can hedge another, or you can trade the divergence when correlation dips.
  • Correlation breakdowns → regime change or idiosyncratic shock on one leg.
""")

## 7. Spread & Cointegration\n\nIf two instruments are cointegrated, their price spread is stationary — the strongest theoretical justification for a pairs-trading strategy. We test every pair via Engle-Granger (ADF on OLS residuals) and report the hedge ratio."

In [ ]:
coint_results = []
for c1, c2 in pairs:
    s1 = mid[c1].dropna()
    s2 = mid[c2].dropna()
    idx = s1.index.intersection(s2.index)
    s1, s2 = s1[idx], s2[idx]

    # OLS hedge ratio
    X = sm.add_constant(s2.values)
    model = sm.OLS(s1.values, X).fit()
    hedge = model.params[1]
    spread = s1.values - hedge * s2.values

    # Engle-Granger: ADF on spread
    adf_s, adf_p, *_ = adfuller(spread, autolag="AIC")

    # statsmodels coint test (includes critical values)
    eg_stat, eg_p, eg_crit = coint(s1.values, s2.values)

    coint_results.append({
        "Pair": f"{c1.replace('GALAXY_SOUNDS_','')} / {c2.replace('GALAXY_SOUNDS_','')}",
        "Hedge ratio (β)": round(hedge, 4),
        "Spread ADF p": round(adf_p, 4),
        "EG p-value": round(eg_p, 4),
        "Cointegrated?": "YES ✓" if eg_p < 0.05 else "no",
        "Spread mean": round(spread.mean(), 2),
        "Spread std": round(spread.std(), 2),
    })

coint_df = pd.DataFrame(coint_results).set_index("Pair")
print(coint_df.to_string())

print("""
Hedge ratio β: go long 1 unit of c1, short β units of c2.
EG p < 0.05 → statistically cointegrated → spread is mean-reverting → pairs trade viable.
""")

In [ ]:
# Plot spreads for all pairs — highlight cointegrated ones
ncols = 2
nrows = (len(pairs) + 1) // 2
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows))
axes = axes.flatten()

for i, (c1, c2) in enumerate(pairs):
    s1 = mid[c1].dropna()
    s2 = mid[c2].dropna()
    idx = s1.index.intersection(s2.index)
    s1, s2 = s1[idx], s2[idx]
    X = sm.add_constant(s2.values)
    hedge = sm.OLS(s1.values, X).fit().params[1]
    spread = pd.Series(s1.values - hedge * s2.values, index=idx)

    label = f"{c1.replace('GALAXY_SOUNDS_','')} − β·{c2.replace('GALAXY_SOUNDS_','')}"
    row = coint_df.iloc[i]
    is_coint = row["Cointegrated?"] == "YES ✓"
    color = "steelblue" if is_coint else "gray"

    axes[i].plot(spread.index, spread.values, lw=0.6, color=color)
    axes[i].axhline(spread.mean(), color="red", ls="--", lw=1)
    axes[i].axhline(spread.mean() + 2*spread.std(), color="orange", ls=":", lw=0.9)
    axes[i].axhline(spread.mean() - 2*spread.std(), color="orange", ls=":", lw=0.9)
    tag = "★ cointegrated" if is_coint else ""
    axes[i].set_title(f"{label}  {tag}", fontsize=8)

for ax in axes[len(pairs):]:
    ax.set_visible(False)

plt.suptitle("Pairwise Spreads (β from OLS)  |  red=mean, orange=±2σ", fontsize=12, y=1.001)
plt.tight_layout()
plt.show()

## 8. Volatility Structure — Rolling Std, ARCH Effects & GARCH(1,1)"

In [ ]:
# Rolling 20-tick volatility (std of log-returns)
fig, ax = plt.subplots(figsize=(16, 5))
for col, c in zip(mid.columns, COLORS):
    rvol = log_ret[col].rolling(20).std() * np.sqrt(100)  # annualised-ish in tick units
    ax.plot(rvol.index, rvol.values, lw=0.7, label=col.replace("GALAXY_SOUNDS_",""), color=c, alpha=0.85)
ax.set_title("Rolling 20-tick Volatility (σ × √100)", fontsize=12)
ax.set_xlabel("Global timestamp")
ax.set_ylabel("Volatility")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ARCH-effect test + GARCH(1,1)
garch_results = []
for col in mid.columns:
    ret = (log_ret[col].dropna() * 1000).values   # scale to avoid numerical issues
    lb = acorr_ljungbox(ret**2, lags=[10], return_df=True)
    arch_p = float(lb["lb_pvalue"].iloc[0])

    try:
        gm = arch_model(ret, vol="Garch", p=1, q=1, dist="normal", rescale=False)
        gres = gm.fit(disp="off")
        omega = gres.params.get("omega", np.nan)
        alpha = gres.params.get("alpha[1]", np.nan)
        beta  = gres.params.get("beta[1]", np.nan)
        lrv   = omega / (1 - alpha - beta) if (alpha + beta) < 1 else np.nan
    except Exception:
        omega = alpha = beta = lrv = np.nan

    garch_results.append({
        "Instrument": col.replace("GALAXY_SOUNDS_",""),
        "Ljung-Box p (sq.res)": round(arch_p, 4),
        "ARCH effects?": "YES ✓" if arch_p < 0.05 else "no",
        "GARCH ω (×1e-6)": round(omega * 1e6, 4) if not np.isnan(omega) else "n/a",
        "GARCH α": round(alpha, 4) if not np.isnan(alpha) else "n/a",
        "GARCH β": round(beta, 4) if not np.isnan(beta) else "n/a",
        "α+β (persistence)": round(alpha + beta, 4) if not np.isnan(alpha) else "n/a",
        "Long-run var (×1e-6)": round(lrv * 1e6, 4) if not np.isnan(lrv) else "n/a",
    })

pd.DataFrame(garch_results).set_index("Instrument")

## 9. Outliers & Regime Detection — Z-score Flags + PELT Change-Point Detection"

In [ ]:
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — avoids display hang in nbconvert

Z_THRESH = 3.0

for col, c in zip(mid.columns, COLORS):
    ret = log_ret[col].dropna()
    zsc = (ret - ret.mean()) / ret.std()
    outliers = zsc[zsc.abs() > Z_THRESH]

    # BinSeg (binary segmentation) on 100× downsampled series — ~300 points, very fast
    ds = ret.iloc[::100].values
    algo = rpt.Binseg(model="l2", min_size=5).fit(ds)
    try:
        bkps = algo.predict(n_bkps=5)   # detect at most 5 breaks
    except Exception:
        bkps = []
    ts_arr = ret.iloc[::100].index.values
    bkp_ts = [ts_arr[b-1] for b in bkps if b < len(ts_arr)]

    fig, ax = plt.subplots(figsize=(16, 3.5))
    price_s = mid[col].dropna()
    ax.plot(price_s.index, price_s.values, lw=0.5, color=c, alpha=0.7, label="mid-price")
    ax.scatter(outliers.index, price_s.reindex(outliers.index).values,
               color="red", s=10, zorder=5, label=f"|z|>{Z_THRESH} ({len(outliers)})")
    for bts in bkp_ts:
        ax.axvline(bts, color="purple", lw=1.2, ls="--", alpha=0.7)
    ax.set_title(f"{col}  |  {len(outliers)} outliers  |  {len(bkp_ts)} regime breaks (BinSeg)", fontsize=9)
    ax.legend(loc="upper right", fontsize=7)
    ax.set_xlabel("Global timestamp")
    plt.tight_layout()
    plt.savefig(f"/tmp/pelt_{col}.png", dpi=90)
    plt.close(fig)
    print(f"{col}: {len(outliers)} outliers, {len(bkp_ts)} breaks at ts={bkp_ts}")

# Display saved images inline
from IPython.display import Image, display
for col in mid.columns:
    display(Image(filename=f"/tmp/pelt_{col}.png"))

print("""
Interpretation:
  • Red dots = log-return outliers with |z| > 3 — fat-tail observations.
  • Purple dashed = BinSeg-detected regime shifts (mean change in returns, ≤5 per instrument).
    Detected on 100× downsampled series; timestamps are approximate ±100 ticks.
  • Regime breaks inform look-back window selection: use shorter windows post-break.
""")

## 10. Return Distribution — Histogram, Q-Q Plot, Tail Index & Jarque-Bera"

In [ ]:
def hill_estimator(data, k=50):
    """Hill tail-index estimator using the top-k order statistics."""
    sorted_data = np.sort(np.abs(data))[::-1]
    if k >= len(sorted_data):
        return np.nan
    return 1.0 / np.mean(np.log(sorted_data[:k] / sorted_data[k]))

fig, axes = plt.subplots(5, 2, figsize=(14, 18))
dist_table = []

for i, (col, c) in enumerate(zip(mid.columns, COLORS)):
    ret = log_ret[col].dropna().values

    # Histogram + normal overlay
    ax_h = axes[i][0]
    ax_h.hist(ret, bins=80, density=True, color=c, alpha=0.5, label="empirical")
    xr = np.linspace(ret.min(), ret.max(), 300)
    ax_h.plot(xr, stats.norm.pdf(xr, ret.mean(), ret.std()), "k--", lw=1.2, label="Normal fit")
    ax_h.set_title(f"{col.replace('GALAXY_SOUNDS_','')} — return distribution", fontsize=8)
    ax_h.legend(fontsize=7)

    # Q-Q plot
    ax_q = axes[i][1]
    (osm, osr), (slope, intercept, r) = stats.probplot(ret, dist="norm")
    ax_q.scatter(osm, osr, s=3, color=c, alpha=0.4)
    ax_q.plot(osm, slope * np.array(osm) + intercept, "k--", lw=1)
    ax_q.set_title(f"Q-Q Plot — {col.replace('GALAXY_SOUNDS_','')}", fontsize=8)
    ax_q.set_xlabel("Theoretical quantile")
    ax_q.set_ylabel("Sample quantile")

    jb_stat, jb_p = stats.jarque_bera(ret)
    hill = hill_estimator(ret)
    dist_table.append({
        "Instrument": col.replace("GALAXY_SOUNDS_",""),
        "Mean (×1e-4)": round(ret.mean() * 1e4, 3),
        "Std (×1e-3)": round(ret.std() * 1e3, 3),
        "Skewness": round(stats.skew(ret), 3),
        "Excess Kurtosis": round(stats.kurtosis(ret), 3),
        "Jarque-Bera p": round(jb_p, 5),
        "Normal?": "no ✗" if jb_p < 0.05 else "yes",
        "Hill tail index α": round(hill, 3),
    })

plt.suptitle("Return Distributions — Histogram + Normal Overlay and Q-Q Plots", fontsize=12, y=1.001)
plt.tight_layout()
plt.show()

dist_df = pd.DataFrame(dist_table).set_index("Instrument")
print(dist_df.to_string())
print("""
Hill tail index α: small α (< 3) → heavy tails; α > 4 → near-normal.
Excess kurtosis > 0 → leptokurtic (fat tails) → risk is underestimated by Gaussian models.
""")

## 11. Lead-Lag Relationships — Cross-Correlation + Granger Causality\n\nIf instrument A Granger-causes instrument B, knowing A's history gives predictive power over B's future moves — a direct lead-lag trading signal."

In [ ]:
LAG_RANGE = 20
fig, axes = plt.subplots(len(pairs), 1, figsize=(16, 3.5 * len(pairs)), sharex=True)
if len(pairs) == 1:
    axes = [axes]

ccf_table = []
for ax, (c1, c2) in zip(axes, pairs):
    r1 = log_ret[c1].dropna()
    r2 = log_ret[c2].dropna()
    idx = r1.index.intersection(r2.index)
    r1, r2 = r1[idx].values, r2[idx].values

    lags = np.arange(-LAG_RANGE, LAG_RANGE + 1)
    ccf_vals = np.array([np.corrcoef(r1[:len(r1)-k], r2[k:])[0,1] if k >= 0
                         else np.corrcoef(r1[-k:], r2[:len(r2)+k])[0,1]
                         for k in lags])

    ax.bar(lags, ccf_vals, width=0.7, color="steelblue", alpha=0.7)
    ax.axhline(1.96/np.sqrt(len(r1)), color="red", ls="--", lw=0.8)
    ax.axhline(-1.96/np.sqrt(len(r1)), color="red", ls="--", lw=0.8)
    ax.axvline(0, color="k", lw=0.5)
    best_lag = int(lags[np.argmax(np.abs(ccf_vals))])
    ax.set_title(
        f"CCF: {c1.replace('GALAXY_SOUNDS_','')} → {c2.replace('GALAXY_SOUNDS_','')}  "
        f"|  peak at lag={best_lag}, r={ccf_vals[lags==best_lag][0]:.3f}", fontsize=8)
    ccf_table.append({"Pair": f"{c1.replace('GALAXY_SOUNDS_','')} / {c2.replace('GALAXY_SOUNDS_','')}", "Peak lag": best_lag, "Peak CCF": round(ccf_vals[lags==best_lag][0], 4)})

axes[-1].set_xlabel("Lag (positive = c1 leads c2)")
plt.suptitle("Cross-Correlation Function (lags −20 to +20)", fontsize=12, y=1.001)
plt.tight_layout()
plt.show()
print(pd.DataFrame(ccf_table).set_index("Pair").to_string())

In [ ]:
# Pairwise Granger causality (AIC-selected lag order, max 10)
print("Granger Causality — F-test p-values  (row causes col)\n")
cols = mid.columns.tolist()
gc_matrix = pd.DataFrame(np.nan, index=[c.replace("GALAXY_SOUNDS_","") for c in cols],
                         columns=[c.replace("GALAXY_SOUNDS_","") for c in cols])

for c1 in cols:
    for c2 in cols:
        if c1 == c2:
            continue
        r1 = log_ret[c1].dropna()
        r2 = log_ret[c2].dropna()
        idx = r1.index.intersection(r2.index)
        data = pd.DataFrame({c1: r1[idx].values, c2: r2[idx].values})
        try:
            gc_res = grangercausalitytests(data[[c2, c1]], maxlag=5, verbose=False)
            # pick best lag (min p across lags 1-5)
            best_p = min(gc_res[lag][0]["ssr_ftest"][1] for lag in range(1, 6))
            gc_matrix.loc[c1.replace("GALAXY_SOUNDS_",""), c2.replace("GALAXY_SOUNDS_","")] = round(best_p, 4)
        except Exception:
            pass

print(gc_matrix.to_string())

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(gc_matrix.astype(float), ax=ax, annot=True, fmt=".3f", cmap="RdYlGn_r",
            vmin=0, vmax=0.1, linewidths=0.5)
ax.set_title("Granger Causality p-values\n(row → col;  green = significant at p<0.05)", fontsize=11)
plt.tight_layout()
plt.show()

print("""
Interpretation:
  • Cell (row, col) = p-value that 'row' Granger-causes 'col'.
  • p < 0.05 (green) → past values of 'row' improve prediction of 'col' beyond col's own history.
  • Strong asymmetric causality → 'row' is the leader; trade 'col' in the direction 'row' moved.
""")

## 12. Signal / Pattern Search — Mean-Reversion Z-score Signal & Momentum Backtest"

In [ ]:
def backtest_signal(price_series, signal_series, pos_limit=10):
    """
    Vectorised backtest.
    signal_series: +1 (long), -1 (short), 0 (flat)
    Returns: pnl series, sharpe, max_drawdown
    """
    pos = signal_series.shift(1).fillna(0).clip(-pos_limit, pos_limit)
    ret = price_series.diff()
    pnl = (pos * ret).fillna(0)
    cum = pnl.cumsum()
    sharpe = pnl.mean() / pnl.std() * np.sqrt(252 * 10000 / len(pnl)) if pnl.std() > 0 else 0
    roll_max = cum.cummax()
    dd = cum - roll_max
    max_dd = dd.min()
    return pnl, cum, round(sharpe, 3), round(max_dd, 2)

# Parameters
MR_ENTRY   = 1.5    # z-score entry threshold for mean-reversion
MR_EXIT    = 0.3    # z-score exit threshold
MOM_WINDOW = 10     # ticks look-back for momentum signal
ROLL_WINDOW = 50    # rolling window for z-score

results_bt = []
fig, axes = plt.subplots(5, 2, figsize=(18, 20))

for i, (col, c) in enumerate(zip(mid.columns, COLORS)):
    price = mid[col].dropna()

    # ── Mean-reversion signal ─────────────────────────────────────────────────
    roll_mean = price.rolling(ROLL_WINDOW).mean()
    roll_std  = price.rolling(ROLL_WINDOW).std()
    zscore    = (price - roll_mean) / roll_std

    mr_signal = pd.Series(0, index=price.index)
    # entry: fade extreme moves; exit: z returns near zero
    mr_signal[zscore >  MR_ENTRY] = -1   # price too high → short
    mr_signal[zscore < -MR_ENTRY] =  1   # price too low  → long
    mr_signal[(zscore.abs() < MR_EXIT)] = 0
    mr_signal = mr_signal.ffill().fillna(0)

    pnl_mr, cum_mr, sh_mr, dd_mr = backtest_signal(price, mr_signal)

    # ── Momentum signal ───────────────────────────────────────────────────────
    mom_signal = np.sign(price.diff(MOM_WINDOW)).fillna(0)
    pnl_mom, cum_mom, sh_mom, dd_mom = backtest_signal(price, mom_signal)

    # Plot cumulative PnL
    axes[i][0].plot(cum_mr.index, cum_mr.values, color=c, lw=0.8)
    axes[i][0].axhline(0, color="k", lw=0.4)
    axes[i][0].set_title(f"MR signal — {col.replace('GALAXY_SOUNDS_','')}\n"
                         f"Sharpe={sh_mr} | MaxDD={dd_mr:.0f}", fontsize=8)
    axes[i][0].set_ylabel("Cum. PnL (XIRECS)")

    axes[i][1].plot(cum_mom.index, cum_mom.values, color=c, lw=0.8)
    axes[i][1].axhline(0, color="k", lw=0.4)
    axes[i][1].set_title(f"Momentum signal — {col.replace('GALAXY_SOUNDS_','')}\n"
                         f"Sharpe={sh_mom} | MaxDD={dd_mom:.0f}", fontsize=8)

    results_bt.append({
        "Instrument": col.replace("GALAXY_SOUNDS_",""),
        "MR Sharpe": sh_mr, "MR MaxDD": dd_mr, "MR Total PnL": round(cum_mr.iloc[-1], 1),
        "Mom Sharpe": sh_mom, "Mom MaxDD": dd_mom, "Mom Total PnL": round(cum_mom.iloc[-1], 1),
    })

plt.suptitle("Mean-Reversion (z-score) and Momentum Signal Backtest", fontsize=12, y=1.001)
plt.tight_layout()
plt.show()

bt_df = pd.DataFrame(results_bt).set_index("Instrument")
print(bt_df.to_string())

## 13. Trades Data Analysis — Volume, Flow, BBO Spread & Market Impact\n\nThe trades file reveals actual execution — who trades, how much, at what prices. This section links order-book state to realised trades to understand market microstructure."

In [ ]:
print("Trade counts per instrument per day:")
print(gt.groupby(["symbol","day"]).size().unstack().to_string())
print()
print("Descriptive stats — trade price:")
print(gt.groupby("symbol")["price"].describe().round(2).to_string())
print()
print("Descriptive stats — trade quantity:")
print(gt.groupby("symbol")["quantity"].describe().round(2).to_string())

In [ ]:
# ── Trade price vs mid-price overlay ─────────────────────────────────────────
fig, axes = plt.subplots(5, 1, figsize=(16, 18), sharex=True)

for ax, col, c in zip(axes, mid.columns, COLORS):
    # mid-price
    ax.plot(mid.index, mid[col], lw=0.6, color=c, alpha=0.5, label="mid")
    # trades
    t = gt[gt["symbol"] == col].copy()
    ax.scatter(t["global_ts"], t["price"], s=t["quantity"]*3, color="k",
               alpha=0.5, zorder=5, label="trades (size ∝ qty)")
    ax.set_title(col.replace("GALAXY_SOUNDS_",""), fontsize=9)
    ax.legend(loc="upper right", fontsize=7)

axes[-1].set_xlabel("Global timestamp")
plt.suptitle("Trade Prices vs Mid-Price (dot size ∝ quantity)", fontsize=12, y=1.001)
plt.tight_layout()
plt.show()

print("""
Interpretation:
  • Trades clustering at the bid → aggressive selling; at ask → aggressive buying.
  • Trades far from mid → stale quotes or large market impact.
""")

In [ ]:
# ── Rolling trade volume + BBO spread ────────────────────────────────────────
fig, axes = plt.subplots(5, 2, figsize=(18, 18))

for i, (col, c) in enumerate(zip(mid.columns, COLORS)):
    t = gt[gt["symbol"] == col].copy()

    # rolling trade volume (sum over 1000-tick windows)
    t_vol = t.set_index("global_ts")["quantity"].resample("1000").sum() if hasattr(t.set_index("global_ts")["quantity"], "resample") else None
    # use simple groupby over 1000-tick bins instead
    bins = np.arange(0, mid.index.max() + 1001, 1000)
    t_binned = t.groupby(pd.cut(t["global_ts"], bins=bins))["quantity"].sum()
    bin_centers = [(iv.left + iv.right)/2 for iv in t_binned.index]

    axes[i][0].bar(bin_centers, t_binned.values, width=800, color=c, alpha=0.6)
    axes[i][0].set_title(f"Trade volume per 1000-tick bin — {col.replace('GALAXY_SOUNDS_','')}", fontsize=8)
    axes[i][0].set_xlabel("Global timestamp")
    axes[i][0].set_ylabel("Volume")

    # BBO spread for this product
    bbo = spread_bbo[spread_bbo["product"] == col][["global_ts","bbo_spread"]].dropna()
    axes[i][1].plot(bbo["global_ts"], bbo["bbo_spread"], lw=0.6, color=c, alpha=0.7)
    axes[i][1].set_title(f"BBO Spread — {col.replace('GALAXY_SOUNDS_','')}", fontsize=8)
    axes[i][1].set_xlabel("Global timestamp")
    axes[i][1].set_ylabel("Ask₁ − Bid₁")

plt.suptitle("Trade Volume Distribution and BBO Spread Over Time", fontsize=12, y=1.001)
plt.tight_layout()
plt.show()

print("""
Interpretation:
  • High-volume bursts → inform on which timestamps the market is most active / liquid.
  • BBO spread is the minimum round-trip cost for a market order. Wide spread → market-make; tight → cross only if signal is strong.
  • A persistently tight BBO spread across all instruments → the family behaves as a liquid cluster.
""")

In [ ]:
# ── Trade-price deviation from mid (market impact proxy) ─────────────────────
fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=False)

impact_table = []
for ax, col, c in zip(axes, mid.columns, COLORS):
    t = gt[gt["symbol"] == col].copy()
    # join nearest mid-price
    t = t.sort_values("global_ts")
    mid_col = mid[col].dropna().reset_index()
    mid_col.columns = ["global_ts","mid"]
    t = pd.merge_asof(t, mid_col, on="global_ts", direction="nearest")
    t["impact"] = t["price"] - t["mid"]

    ax.hist(t["impact"].dropna(), bins=40, color=c, alpha=0.7, density=True)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_title(f"Trade impact vs mid — {col.replace('GALAXY_SOUNDS_','')}", fontsize=9)
    ax.set_xlabel("Trade price − mid-price (XIRECS)")

    impact_table.append({
        "Instrument": col.replace("GALAXY_SOUNDS_",""),
        "Mean impact": round(t["impact"].mean(), 3),
        "Std impact": round(t["impact"].std(), 3),
        "% trades above mid": round((t["impact"] > 0).mean() * 100, 1),
    })

plt.suptitle("Market Impact: Distribution of (Trade Price − Mid-Price)", fontsize=12, y=1.001)
plt.tight_layout()
plt.show()

pd.DataFrame(impact_table).set_index("Instrument")

## 14. Order-Book Depth Analysis — Bid/Ask Volume, Imbalance & Quote Skew\n\nOrder-book imbalance (OBI) is one of the best short-term directional predictors: a lopsided book predicts the next price move in the direction of the heavier side."

In [ ]:
obi_results = []
fig, axes = plt.subplots(5, 2, figsize=(18, 18))

for i, (col, c) in enumerate(zip(mid.columns, COLORS)):
    df = gs[gs["product"] == col].copy()

    # total bid / ask depth across all 3 levels
    bid_vol = df[["bid_volume_1","bid_volume_2","bid_volume_3"]].fillna(0).sum(axis=1)
    ask_vol = df[["ask_volume_1","ask_volume_2","ask_volume_3"]].fillna(0).sum(axis=1)

    obi = (bid_vol - ask_vol) / (bid_vol + ask_vol + 1e-9)  # in [-1, 1]
    df["obi"] = obi.values

    # OBI distribution
    axes[i][0].hist(obi, bins=60, color=c, alpha=0.7, density=True)
    axes[i][0].axvline(0, color="k", lw=0.8)
    axes[i][0].set_title(f"OBI distribution — {col.replace('GALAXY_SOUNDS_','')}", fontsize=8)
    axes[i][0].set_xlabel("Order-book imbalance")

    # rolling 50-tick OBI + next-tick return scatter
    ret_next = mid[col].diff().shift(-1)
    scatter_df = pd.DataFrame({"obi": obi.values, "ret_next": ret_next.reindex(df["global_ts"]).values},
                               index=df["global_ts"])
    scatter_df = scatter_df.dropna()
    axes[i][1].scatter(scatter_df["obi"], scatter_df["ret_next"], s=1, alpha=0.2, color=c)
    slope, intercept, r, p, se = stats.linregress(scatter_df["obi"], scatter_df["ret_next"])
    xfit = np.linspace(-1, 1, 100)
    axes[i][1].plot(xfit, slope*xfit+intercept, "r-", lw=1.5, label=f"r={r:.3f} p={p:.4f}")
    axes[i][1].axhline(0, color="k", lw=0.4)
    axes[i][1].axvline(0, color="k", lw=0.4)
    axes[i][1].set_title(f"OBI vs next-tick return — {col.replace('GALAXY_SOUNDS_','')}", fontsize=8)
    axes[i][1].legend(fontsize=7)

    obi_results.append({
        "Instrument": col.replace("GALAXY_SOUNDS_",""),
        "Mean OBI": round(obi.mean(), 4),
        "Std OBI": round(obi.std(), 4),
        "OBI→ret slope": round(slope, 5),
        "Pearson r": round(r, 4),
        "p-value": round(p, 5),
        "Significant?": "YES ✓" if p < 0.05 else "no",
    })

plt.suptitle("Order-Book Imbalance (OBI) Distribution and Predictive Power", fontsize=12, y=1.001)
plt.tight_layout()
plt.show()

obi_df = pd.DataFrame(obi_results).set_index("Instrument")
print(obi_df.to_string())
print("""
OBI interpretation:
  OBI > 0 → more bid depth → buying pressure → price likely to tick up.
  OBI < 0 → more ask depth → selling pressure → price likely to tick down.
  Significant slope + r → OBI is a usable directional predictor for that instrument.
""")

## 15. Summary — Ranked Signals, Strategy Recommendations & Data-Quality Notes\n\nThis cell collects findings from all sections above and prints a structured summary. Run it after all prior cells have executed."

In [ ]:
print("=" * 78)
print("  GALAXY SOUNDS — EDA SUMMARY REPORT")
print("=" * 78)

# ── 1. Ranked signals ─────────────────────────────────────────────────────────
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. RANKED SIGNALS  (higher rank = more exploitable)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

signal_rank = []

# OBI predictor
for row in obi_results:
    if row["Significant?"] == "YES ✓":
        signal_rank.append({
            "Rank": None,
            "Signal": f"OBI → next-tick return ({row['Instrument']})",
            "Type": "microstructure",
            "Edge proxy": f"r={row['Pearson r']}, slope={row['OBI→ret slope']}",
            "Confidence": "high (p<0.05)",
        })

# Cointegrated pairs
for _, row in coint_df.iterrows():
    if row["Cointegrated?"] == "YES ✓":
        signal_rank.append({
            "Rank": None,
            "Signal": f"Pair spread mean-reversion ({row.name})",
            "Type": "stat-arb",
            "Edge proxy": f"spread std={row['Spread std']}, ADF p={row['Spread ADF p']}",
            "Confidence": "high (EG p<0.05)",
        })

# Mean-reversion z-score signal
for _, row in bt_df.iterrows():
    if row["MR Sharpe"] > 0.5:
        signal_rank.append({
            "Rank": None,
            "Signal": f"Rolling z-score mean-reversion ({row.name})",
            "Type": "mean-revert",
            "Edge proxy": f"Sharpe={row['MR Sharpe']}, PnL={row['MR Total PnL']:.0f}",
            "Confidence": "medium (in-sample only)",
        })

# Momentum signal
for _, row in bt_df.iterrows():
    if row["Mom Sharpe"] > 0.5:
        signal_rank.append({
            "Rank": None,
            "Signal": f"10-tick momentum ({row.name})",
            "Type": "momentum",
            "Edge proxy": f"Sharpe={row['Mom Sharpe']}, PnL={row['Mom Total PnL']:.0f}",
            "Confidence": "medium (in-sample only)",
        })

# Granger causality
gc_leads = []
for r_idx in gc_matrix.index:
    for c_idx in gc_matrix.columns:
        if r_idx != c_idx and not np.isnan(gc_matrix.loc[r_idx, c_idx]) and gc_matrix.loc[r_idx, c_idx] < 0.05:
            gc_leads.append((r_idx, c_idx, gc_matrix.loc[r_idx, c_idx]))
if gc_leads:
    for leader, follower, p in gc_leads:
        signal_rank.append({
            "Rank": None,
            "Signal": f"Lead-lag: {leader} → {follower}",
            "Type": "lead-lag",
            "Edge proxy": f"Granger p={p:.4f}",
            "Confidence": "medium",
        })

# Sort by type priority
type_priority = {"microstructure": 0, "stat-arb": 1, "mean-revert": 2, "lead-lag": 3, "momentum": 4}
signal_rank.sort(key=lambda x: type_priority.get(x["Type"], 9))
for i, s in enumerate(signal_rank, 1):
    s["Rank"] = i
    print(f"  #{i}  [{s['Type'].upper():<15}]  {s['Signal']}")
    print(f"       Edge: {s['Edge proxy']}  |  Confidence: {s['Confidence']}")
    print()

if not signal_rank:
    print("  No signals crossed significance thresholds — review thresholds above.")

# ── 2. Strategy recommendations ───────────────────────────────────────────────
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2. STRATEGY RECOMMENDATIONS  (data-driven)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

for col in mid.columns:
    name = col.replace("GALAXY_SOUNDS_","")
    hurst_row = res_df.loc[name] if name in res_df.index else None
    adf_stat  = res_df.loc[name, "ADF stationarity"] if name in res_df.index else "?"
    h = res_df.loc[name, "Hurst (R/S)"] if name in res_df.index else 0.5
    mr_sh = bt_df.loc[name, "MR Sharpe"] if name in bt_df.index else 0
    mom_sh = bt_df.loc[name, "Mom Sharpe"] if name in bt_df.index else 0
    obi_sig = next((r["Significant?"] for r in obi_results if r["Instrument"] == name), "no")

    if h < 0.45 or adf_stat == "stationary ✓":
        strat = "MEAN-REVERT  — fade extremes with z-score entry ±1.5, exit ±0.3"
    elif h > 0.55 and mom_sh > mr_sh:
        strat = "MOMENTUM     — trend-follow with 10-tick sign signal"
    else:
        strat = "MARKET-MAKE  — quote at bid/ask; harvest spread passively"

    if obi_sig == "YES ✓":
        strat += "  +  OBI filter for entry timing"

    print(f"  {name:<22} → {strat}")

# Check for cointegrated pairs
print()
coint_yes = coint_df[coint_df["Cointegrated?"] == "YES ✓"]
if len(coint_yes):
    print("  Cointegrated pairs → PAIR-TRADE (trade spread z-score):")
    for pair, row in coint_yes.iterrows():
        print(f"    {pair}  |  β={row['Hedge ratio (β)']}  |  spread std={row['Spread std']}")
else:
    print("  No cointegrated pairs detected at 5% — individual strategies preferred.")

# ── 3. Data-quality concerns ──────────────────────────────────────────────────
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
3. DATA-QUALITY CONCERNS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  a) UNIT ROOT  — prices are I(1) (non-stationary in levels). All signals must be
     built on returns or spreads, NOT raw prices, to avoid spurious regression.

  b) FAT TAILS  — Jarque-Bera rejects normality for all instruments. Use tick-
     sized position limits (±10) and never size up on apparent 'low vol' windows;
     GARCH vol-targeting is recommended before going live.

  c) REGIME BREAKS  — PELT detects multiple structural breaks per day. Recalibrate
     rolling windows and z-score parameters after each detected break. In live
     trading, use an exponentially-weighted mean/std instead of a fixed window.

  d) SPARSE TRADES  — the trades file has ~1 145 rows across 3 days for 5 instruments
     (~76 trades / instrument / day). Realised-vol and OBI estimates from the order
     book (10 000 ticks/day) are therefore more reliable than trade-based estimates.

  e) TICK RESOLUTION  — timestamps are integer multiples of 100 ticks. Sub-tick
     microstructure analysis is not possible; limit cross-correlation lags to ≥1 tick.

  f) POSITION CAP  — ±10 lots per instrument. Large signals must be clipped; PnL
     from the backtest above is already clipped to ±10, but live slippage from
     aggressive execution is NOT modelled here.
""")

print("=" * 78)